In [ ]:
import pandas as pd
import numpy as np 
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))


## Load reliable cohorts and representative cohorts

In [ ]:
from config.constants import PROJECT_ROOT
cohorts_dir = PROJECT_ROOT / "saved_data" / "cohorts" / "DTB" / "new"

cohorts_init = pd.read_csv(cohorts_dir / "initial_cohorts_summary.csv")
cohorts_first_adms = pd.read_csv(cohorts_dir / "first_adms_cohorts_summary.csv")
reliable_cohorts_init = pd.read_csv(cohorts_dir / "reliable_cohorts_init.csv", header=None, names=["cohort_name"])
reliable_cohorts_first_adms = pd.read_csv(cohorts_dir / "reliable_cohorts_first_adms.csv", header=None, names=["cohort_name"])
cohorts_first_adms_reliable = cohorts_first_adms[cohorts_first_adms["cohort_name"].isin(reliable_cohorts_first_adms["cohort_name"])]


rep_cohorts = pd.read_csv(cohorts_dir / "representative_cohorts_df.csv").cohort_name


In [ ]:
selected_cohorts = set(rep_cohorts) #& set(reliable_cohorts_first_adms.cohort_name)

## Add pr-gain & pr-skill

In [ ]:

#results_dir = PROJECT_ROOT / "saved_data" / "results"
def add_pr_metrics(df):
    out = df.copy()
    out["pr_lift"] = out["auprc"] / out["target_rate"]
    out["pr_skill"] = (out["auprc"] - out["target_rate"]) / (1 - out["target_rate"])
    return out

## Load results

In [ ]:
# Loading results
def load_results(cohorts, models, prefixes, fold=None):

    folds = [fold] if fold is not None else range(5)
    rows = []
    for cohort in cohorts:
        for model in models:
            prefix = next((p for p in prefixes if (Path(f"../saved_data/hpc_run_3008/results/{cohort}/{model}") / p).exists()),None)
            if prefix is None:
                print("no prefix")
                continue
            for f in folds:
                file = (Path(f"../saved_data/hpc_run_3008/results/{cohort}/{model}/{prefix}")
                        / f"fold_{f}" / "agg_int_24" / "impute_fill" / "variant_VMD" / "results_final.csv")
                if not file.exists():
                    print(f"{cohort} - fold_{f} doesn't exist")
                    continue
                output = pd.read_csv(file)
                output["cohort_name"] = cohort
                output["model"] = model
                output["prefix"] = prefix
                output["fold"] = f
                rows.append(output)
                
    results_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    results_df = results_df[["cohort_name","model","prefix","fold","auroc", "auprc", "f1","mcc","recall","precision"]]
    
    for col in ["auroc", "auprc", "f1","mcc","recall","precision"]:
        results_df[col] = pd.to_numeric(results_df[col], errors="coerce")

    #for col, new_col in [("auroc", "roc_avg"), ("auprc", "pr_avg"), ("f1", "f1_avg"),("mcc", "mcc_avg"),("recall","recall_avg"),("precision","precision_avg")]:
    #    results_df[new_col] = results_df.groupby(["cohort_name", "model"])[col].transform("mean") # average of folds
        
    #results_df = results_df[["cohort_name", "model", "roc_avg", "pr_avg", "f1_avg","mcc_avg","recall_avg","precision_avg"]].drop_duplicates(["cohort_name", "model"])
    return results_df



In [ ]:
models = ["random_forest"] #"catboost", "random_forest", "xgboost"]


In [ ]:
VARIANT_CONFIG = {
    "Initial_14d":     (["200526", "190526"],   cohorts_init),
    "Corr-top100_14d": (["corr_feat_training"], cohorts_init),
    "MRMR_14d":        (["mrmr_feat_training"], cohorts_init),
    "first_adms_14d":  (["first_adms"],         cohorts_first_adms),
    "first_adms_90d":  (["first_adms_90d"],     cohorts_first_adms),
}

results = {}
for name, (prefixes, cohort_chars) in VARIANT_CONFIG.items():
    print("Loading,", name)
    df = load_results(selected_cohorts, models, prefixes, fold=0)
    df = df.merge(cohort_chars, on="cohort_name")
    df = add_pr_metrics(df)
    results[name] = df

results_initial       = results["Initial_14d"]
results_corr_100      = results["Corr-top100_14d"]
results_mrmr           = results["MRMR_14d"]
results_first_adms     = results["first_adms_14d"]
results_first_adms_90d = results["first_adms_90d"]


## Compare performance across different runs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

metrics = [
    ("roc_avg", "AUROC", "Random Forest AUROC"),
    #("pr_avg", "AUPRC", "Random Forest AUPRC"),
    ("pr_skill", "PR_norm", "Random Forest PR_norm"),
    ("f1_avg", "F1", "Random Forest F1"),
]
labels = ["Initial_14d", "Corr-top100_14d", "MRMR_14d", "first_adms_14d", "first_adms_90d"]
colors = ["#2a78d6", "#008300", "#e87ba4", "#833b1d", "#ed4b06"]
dfs = [results_initial, results_corr_100, results_mrmr, results_first_adms, results_first_adms_90d]

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 7), sharey=False)

for ax, (col, ylabel, title) in zip(axes, metrics):
    variants = [df[col].dropna() for df in dfs]

    bp = ax.boxplot(variants, tick_labels=labels, widths=0.5, patch_artist=True,
                     showfliers=False, medianprops=dict(color="black", linewidth=2))

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.2)
        patch.set_edgecolor(color)
        patch.set_linewidth(2)

    for i, (df, vals, color) in enumerate(zip(dfs, variants, colors), start=1):
        rel = df.loc[vals.index, "reliable"].values if "reliable" in df.columns else np.ones(len(vals), dtype=bool)

        # dodge: reliable points to the left of center, unreliable to the right
        jitter_rel = i - 0.15 + np.random.normal(0, 0.045, size=rel.sum())
        jitter_unrel = i + 0.15 + np.random.normal(0, 0.045, size=(~rel).sum())

        ax.scatter(jitter_rel, vals.values[rel], color=color, alpha=0.8, s=32, zorder=3, marker="o")
        ax.scatter(jitter_unrel, vals.values[~rel], color=color, alpha=0.5, s=45, zorder=3,
                   marker="x", linewidths=1.8)

    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_title(title, fontsize=16)
    ax.tick_params(axis="x", rotation=30, labelsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='gray', markersize=9, label='Reliable'),
    Line2D([0], [0], marker='x', color='gray', markersize=9, linewidth=0, markeredgewidth=1.8, label='Unreliable'),
]
fig.legend(handles=legend_elems, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.08), fontsize=18)

fig.suptitle("Feature selection & admission variants", y=1.16, fontsize=20)

plt.tight_layout()
plt.show()


## Check if the performance differs across reliable cohorts only

comparing reliable vs unreliable
Mann-Whitney U test — the non-parametric equivalent of a t-test, comparing two independent groups' medians without assuming a normal distribution 

(appropriate here since AUROC/PR-skill values across cohorts are not normally distributed and group sizes differ

In [ ]:
from scipy.stats import mannwhitneyu

def compare_reliable_vs_excluded(df, col, alternative="greater", alpha=0.05):
    reliable = df.loc[df.reliable == True, col].dropna()
    excluded = df.loc[df.reliable == False, col].dropna()

    stat, p = mannwhitneyu(reliable, excluded, alternative=alternative)

    n1, n2 = len(reliable), len(excluded)
    effect = stat / (n1 * n2)   # 0.5 = no difference, 1.0 = every reliable > every excluded

    print(f"{col}: reliable median={reliable.median():.3f} (n={n1})  "
          f"excluded median={excluded.median():.3f} (n={n2})")
    print(f"  P(random reliable > random excluded) = {effect:.3f}")
    print(f"  one-sided p (reliable > excluded) = {p:.4g}")


    if p < alpha:
        diff = abs(effect - 0.5)
        size = "large" if diff >= 0.2 else "moderate" if diff >= 0.1 else "small"
        print(f"  -> CONCLUSION: reliable cohorts have significantly higher {col} "
              f"than excluded ones ({size} effect).")
    else:
        print(f"  -> CONCLUSION: no significant evidence that reliable cohorts have "
              f"higher {col} than excluded ones (p={p:.3g} >= {alpha}).")
    print()

    return effect, p

for col in ["roc_avg", "pr_skill"]:
    compare_reliable_vs_excluded(results_first_adms_90d, col)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

def compare_reliable_vs_excluded(df, col, alternative="greater", alpha=0.05, verbose=True):
    reliable = df.loc[df.reliable == True, col].dropna()
    excluded = df.loc[df.reliable == False, col].dropna()
    if len(reliable) < 2 or len(excluded) < 2:
        return None

    stat, p = mannwhitneyu(reliable, excluded, alternative=alternative)
    n1, n2 = len(reliable), len(excluded)
    effect = stat / (n1 * n2)

    if verbose:
        print(f"{col}: reliable median={reliable.median():.3f} (n={n1})  "
              f"excluded median={excluded.median():.3f} (n={n2})")
        print(f"  P(random reliable > random excluded) = {effect:.3f}")
        print(f"  one-sided p (reliable > excluded) = {p:.4g}")
        if p < alpha:
            diff = abs(effect - 0.5)
            size = "large" if diff >= 0.2 else "moderate" if diff >= 0.1 else "small"
            print(f"  -> CONCLUSION: reliable cohorts have significantly higher {col} "
                  f"than excluded ones ({size} effect).")
        else:
            print(f"  -> CONCLUSION: no significant evidence that reliable cohorts have "
                  f"higher {col} than excluded ones (p={p:.3g} >= {alpha}).")
        print()

    return dict(effect=effect, p=p, n_reliable=n1, n_excluded=n2,
                median_reliable=reliable.median(), median_excluded=excluded.median())


variant_dfs = {
    "Initial_14d": results_initial,
    "Corr-top100_14d": results_corr_100,
    "MRMR_14d": results_mrmr,
    "first_adms_14d": results_first_adms,
    "first_adms_90d": results_first_adms_90d,
}

metrics = ["roc_avg", "pr_skill"]

rows = []
for name, df in variant_dfs.items():
    print(f"=== {name} ===")
    for col in metrics:
        res = compare_reliable_vs_excluded(df, col)
        if res:
            rows.append({"variant": name, "metric": col, **res})

summary = pd.DataFrame(rows)

# --- figure: effect size per variant, per metric, with significance stars ---
fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4), sharey=True)
if len(metrics) == 1:
    axes = [axes]

for ax, col in zip(axes, metrics):
    sub = summary[summary.metric == col].reset_index(drop=True)
    x = np.arange(len(sub))
    bar_colors = ["#2a78d6" if p < 0.05 else "#bbbbbb" for p in sub["p"]]

    ax.bar(x, sub["effect"], color=bar_colors, alpha=0.85, edgecolor="black", linewidth=0.8)
    ax.axhline(0.5, color="black", linestyle="--", linewidth=1)

    for xi, (eff, p) in enumerate(zip(sub["effect"], sub["p"])):
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        ax.text(xi, eff + 0.02, stars, ha="center", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(sub["variant"], rotation=30, ha="right")
    ax.set_ylabel("P(reliable > excluded)")
    ax.set_title(col)
    ax.set_ylim(0.3, 1.0)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

fig.suptitle("Reliable vs Excluded cohorts, per variant", y=1.03)
plt.tight_layout()
plt.show()


## Check which models have genuinely learned sth

Flag cohorts whose AUROC is statistically distinguishable from chance (0.5),

using Hanley-McNeil SE + FDR correction across cohorts (avoids trusting noisy

small-N point estimates as if they were real signal).


In [ ]:
import numpy as np
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

def add_significance(df, auc_col="roc_avg", alpha=0.05, correct=True):
    df = df.copy()
    n_pos, n_neg, auc = df["n_pos_test_estimate"], df["n_neg_test_estimate"], df[auc_col]

    Q1 = auc / (2 - auc)
    Q2 = 2 * auc**2 / (1 + auc)
    var = (auc*(1-auc) + (n_pos-1)*(Q1-auc**2) + (n_neg-1)*(Q2-auc**2)) / (n_pos*n_neg)
    se = np.sqrt(var)

    z = (auc - 0.5) / se
    p = pd.Series(1 - norm.cdf(z), index=df.index) 

    df["auroc_se"] = se
    df["auroc_p"] = p
    if correct:
        valid = p.notna()
        df.loc[valid, "auroc_p_adj"] = multipletests(p[valid], method="fdr_bh")[1]
        df["learned"] = df["auroc_p_adj"] < alpha
    else:
        df["learned"] = df["auroc_p"] < alpha
    return df


results_initial = add_significance(results_initial)
results_corr_100 = add_significance(results_corr_100)
results_mrmr = add_significance(results_mrmr)
results_first_adms = add_significance(results_first_adms)
results_first_adms_90d = add_significance(results_first_adms_90d)




In [ ]:
def add_learned(df, alpha=0.05, lift_threshold=2.0):
    # gate 1 - statistical: distinguishable from chance (your existing test, real counts)
    n_pos, n_neg, auc = df["n_pos_test_estimate"], df["n_neg_test_estimate"], df["roc_avg"]
    Q1 = auc/(2-auc); Q2 = 2*auc**2/(1+auc)
    se = np.sqrt((auc*(1-auc) + (n_pos-1)*(Q1-auc**2) + (n_neg-1)*(Q2-auc**2))/(n_pos*n_neg))
    p = pd.Series(1 - norm.cdf((auc - 0.5)/se), index=df.index)
    df["auroc_se"] = se
    ok = p.notna()
    df.loc[ok, "auroc_p_adj"] = multipletests(p[ok], method="fdr_bh")[1]
    df["above_chance"] = df["auroc_p_adj"] < alpha        # <- rename: this is NOT "learned"

    # gate 2 - practical: the discrimination is actually usable
    df["useful"] = df["pr_lift"] >= lift_threshold

    df["learned"] = df["above_chance"] & df["useful"]
    return df

results_initial = add_learned(results_initial)
results_corr_100 = add_learned(results_corr_100)
results_mrmr = add_learned(results_mrmr)
results_first_adms = add_learned(results_first_adms)
results_first_adms_90d = add_learned(results_first_adms_90d)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

def plot_forest_panel(ax, df, auc_col="roc_avg", title="", show_labels=True):
    d = df.dropna(subset=[auc_col, "auroc_se"]).sort_values(auc_col).reset_index(drop=True)
    y = np.arange(len(d))
    ci = 1.96 * d["auroc_se"]

    groups = [
        (True,  True,  "#1a43d9", "#1a43d9", "o"),   # learned, reliable
        (True,  False, "#1a43d9", "#7b7874", "o"),   # learned, unreliable
        (False, True,  "#7b7874", "#1a43d9", "x"),   # not learned, reliable
        (False, False, "#7b7874", "#7b7874", "x"),   # not learned, unreliable
    ]

    for learned_flag, reliable_flag, mcolor, lcolor, marker in groups:
        mask = ((d["learned"] == learned_flag) & (d["reliable"] == reliable_flag)).values
        if mask.sum() == 0:
            continue
        ax.errorbar(d.loc[mask, auc_col], y[mask], xerr=ci[mask], fmt=marker,
                    ecolor=lcolor, mfc=mcolor, mec=mcolor, ms=10, elinewidth=4,
                    capsize=4, capthick=4, mew=2.5)

    ax.axvline(0.5, color="#c7c2bc", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(d["cohort_name"] if show_labels else [], fontsize=24)
    ax.tick_params(axis="x", labelsize=24)
    ax.set_xlabel("AUROC (95% CI)", fontsize=28)
    ax.set_title(title, fontsize=28)


variant_dfs = {
    "Initial_14d": results_initial,
    "Corr-top100_14d": results_corr_100,
    "MRMR_14d": results_mrmr,
    "first_adms_14d": results_first_adms,
    "first_adms_90d": results_first_adms_90d,
}

max_n = max(len(df.dropna(subset=["roc_avg", "auroc_se"])) for df in variant_dfs.values())
fig_height = max(8, max_n * 0.5)

fig, axes = plt.subplots(1, len(variant_dfs), figsize=(6 * len(variant_dfs), fig_height))

for i, (ax, (name, df)) in enumerate(zip(axes, variant_dfs.items())):
    plot_forest_panel(ax, df, auc_col="roc_avg", title=name, show_labels=(i == 0))

legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor="#1a43d9", markeredgecolor="#1a43d9", markersize=14, label='Learned'),
    Line2D([0], [0], marker='x', color="#7b7874", markersize=14, markeredgewidth=2.5, label='Not learned'),
    Line2D([0], [0], color="#1a43d9", linewidth=4, label='Reliable (CI line)'),
    Line2D([0], [0], color="#7b7874", linewidth=4, label='Unreliable (CI line)'),
]
fig.legend(handles=legend_elems, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.03), fontsize=28)

plt.tight_layout()
plt.show()


In [ ]:
reliable_cohorts_first_adms

# Check the learned and not learned cohorts

In [ ]:
# Load folds pos and neg
import pickle

def load_fold_counts(cohorts, fold=0, first_adm_only=False, seed=42):
    suffix = "_firstadm" if first_adm_only else ""
    rows = []
    for c in cohorts:
        f = PROJECT_ROOT / "saved_data" / "folds" / c / f"seed_{seed}{suffix}" / f"fold_{fold}.pkl"
        if not f.exists():
            continue
        with open(f, "rb") as fh:
            _, _, test_ids = pickle.load(fh)
        y = pd.read_csv(PROJECT_ROOT / "saved_data" / "cohorts" / "DTB" / "new" / f"{c}.csv.gz",
                        usecols=["hadm_id", "label"]).set_index("hadm_id")["label"]
        lbl = y.reindex(test_ids[:, 1])
        rows.append({"cohort_name": c,
                     "test_pos_fold": int(lbl.sum()),
                     "test_neg_fold": int((lbl == 0).sum())})
    return pd.DataFrame(rows)

fold_pos_neg = load_fold_counts(selected_cohorts)                          # 14d / all admissions
fold_pos_neg_fa = load_fold_counts(selected_cohorts, first_adm_only=True)  # first_adms variants


In [ ]:
df = results_initial
fold_file = fold_pos_neg

In [ ]:
cols = ['cohort_name', 'roc_avg','pr_skill', 'f1_avg', 'n_cohort', 'n_pos', 'n_neg',
        'target_rate', 'n_pos_test_estimate', 'n_neg_test_estimate']


In [ ]:
lr_cohorts = df[(df.reliable == True) & (df.learned == True)][cols].sort_values("roc_avg", ascending = False).merge(fold_file, on = "cohort_name")
l_nr_cohorts = df[(df.reliable == False) & (df.learned == True)][cols].sort_values("roc_avg", ascending = False).merge(fold_file, on = "cohort_name")
nl_r_cohorts = df[(df.reliable == True) & (df.learned == False)][cols].sort_values("roc_avg", ascending = False).merge(fold_file, on = "cohort_name")
nl_nr_cohorts = df[(df.reliable == False) & (df.learned == False)][cols].sort_values("roc_avg", ascending = False).merge(fold_file, on = "cohort_name")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

groups = {
    "lr":     lr_cohorts,
    "l-nr":   l_nr_cohorts,
    "nl-r":  nl_r_cohorts,
    "nl-nr": nl_nr_cohorts,
}
g = pd.concat([d.assign(group=k) for k, d in groups.items()], ignore_index=True)
g["group"] = pd.Categorical(g["group"], categories=list(groups), ordered=True)
print(g.groupby("group").size())


PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # fixed order, never cycled
SURFACE, INK, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e"

metrics = [("n_cohort", "Cohort size"), ("n_pos", "Positives"), ("n_neg", "Negatives"),
           ("test_pos_fold", "Test positives"), ("test_neg_fold", "Test negatives")]
metrics = [(c, lab) for c, lab in metrics if c in g.columns]

fig, axes = plt.subplots(1, len(metrics), figsize=(3.4 * len(metrics), 5.2), facecolor=SURFACE)
rng = np.random.default_rng(0)

for ax, (col, label) in zip(axes, metrics):
    data = [g.loc[g.group == k, col].dropna().values for k in groups]

    bp = ax.boxplot(data, widths=0.52, patch_artist=True, showfliers=False,
                    medianprops=dict(color=INK, linewidth=2),
                    whiskerprops=dict(color=INK_MUTED, linewidth=1),
                    capprops=dict(color=INK_MUTED, linewidth=1))
    for patch, c in zip(bp["boxes"], PALETTE):
        patch.set(facecolor=c, alpha=0.18, edgecolor=c, linewidth=2)

    for i, (vals, c) in enumerate(zip(data, PALETTE), start=1):
        x = i + rng.normal(0, 0.055, size=len(vals))
        ax.scatter(x, vals, s=34, color=c, alpha=0.85, zorder=3,
                   edgecolor=SURFACE, linewidth=1.4)          # 2px surface ring
        if len(vals):                                          # selective direct label, not every point
            ax.text(i, np.median(vals) * 1.55, f"{np.median(vals):,.0f}",
                    ha="center", fontsize=11, color=INK, fontweight="semibold")

    ax.set_yscale("log")
    ax.set_title(label, fontsize=14, color=INK, pad=10)
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels([f"{k}\n(n={len(d)})" for k, d in zip(groups, data)],
                       fontsize=10, color=INK_MUTED)
    ax.grid(axis="y", linestyle="--", alpha=0.25, color=INK_MUTED)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=INK_MUTED, linewidth=1)
    ax.tick_params(axis="y", labelsize=11, colors=INK_MUTED)
    ax.set_facecolor(SURFACE)

axes[0].set_ylabel("count (log scale)", fontsize=12, color=INK)
fig.suptitle("Cohort composition by learned × reliable", fontsize=16, color=INK, y=1.0)
fig.tight_layout()
plt.show()
#

## new results

In [ ]:
cohorts_cc = pd.read_csv("../saved_data/cohorts/DTB/new/selected_edges_DTB_all.csv")
cohorts_cc["cohort_name"] = cohorts_cc["D1"] + "-" + cohorts_cc["D2"]

In [ ]:
import pandas as pd
rep_name = "representative_cohorts_K100_ab4b58.txt"
rep_cohorts = pd.read_csv(f"../saved_data/hpc_run_3008/rep_selections/{rep_name}",header=None,names=["cohort_name"])


In [ ]:
models = ["random_forest"]

In [ ]:
results_mimic_top = load_results(rep_cohorts["cohort_name"], models,["first_adms_14d_mimic100"] , fold=0)
results_mrmr_top = load_results(rep_cohorts["cohort_name"], models,["first_adms_14d_mrmr100"] , fold=0)

In [ ]:
results_mimic_top = results_mimic_top.merge(cohorts_cc, on = "cohort_name")
results_mrmr_top = results_mrmr_top.merge(cohorts_cc, on = "cohort_name")

In [ ]:
import matplotlib.pyplot as plt

metric = "auroc"
df = results_mrmr_top.sort_values(metric, ascending=False)

fig, ax = plt.subplots(figsize=(16, 5), constrained_layout=True)
for mod, g in df.groupby("model", sort=False):
    ax.plot(g["cohort_name"], g[metric], marker="o", ms=4, lw=1.5, label=mod)

ax.axhline(0.5, color="#52514e", lw=1, ls=(0, (4, 3)))      # chance
ax.set_xticks(range(len(df["cohort_name"].unique())))
ax.set_xticklabels(df["cohort_name"].unique(), rotation=90, fontsize=6)
ax.set_ylabel("ROC AUC"); ax.set_xlabel("cohort")
ax.margins(x=0.005); ax.grid(axis="y", lw=0.8, alpha=0.5); ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.show()


In [ ]:
import pandas as pd
from pathlib import Path

COHORT_DIR   = Path("../saved_data/cohorts/DTB/new")
FEATURES_DIR = Path("../saved_data/features_14d") 

def cohort_prevalence(cohort_name, first_adm_only=True):
    f = COHORT_DIR / f"{cohort_name}.csv.gz"
    if not f.exists():
        return None
    df = pd.read_csv(f, compression="gzip",
                     usecols=["subject_id", "hadm_id", "label", "admittime"])
    #n_raw = len(df)

    # 1. drop admissions with no extracted features (same as generate_folds)
    feat = FEATURES_DIR / cohort_name / "features.csv.gz"
    if feat.exists():
        keep = pd.read_csv(feat, usecols=["hadm_id"])["hadm_id"].unique()
        df = df[df["hadm_id"].isin(keep)]
    n_with_feats = len(df)

    # 2. first admission per patient
    if first_adm_only:
        df = df.sort_values("admittime").drop_duplicates("subject_id", keep="first")

    n_pos = int(df["label"].sum())
    return {
        "cohort_name": cohort_name,
        #"n_raw_adm": n_raw,
        #"n_with_features": n_with_feats,
        "n": len(df),
        "n_subjects": df["subject_id"].nunique(),
        "n_pos": n_pos,
        "n_neg": len(df) - n_pos,
        "target_rate": n_pos / len(df) if len(df) else float("nan"),
    }

cohorts = rep_cohorts["cohort_name"] if hasattr(rep_cohorts, "columns") else rep_cohorts
rows = [r for c in cohorts if (r := cohort_prevalence(c)) is not None]
prev = pd.DataFrame(rows)

missing = set(cohorts) - set(prev.cohort_name)
if missing:
    print(f"{len(missing)} cohort files not found: {sorted(missing)[:5]}")




In [ ]:
results_mimic_top = results_mimic_top.merge(prev, on="cohort_name")
results_mrmr_top = results_mrmr_top.merge(prev, on="cohort_name")

In [ ]:
results_mimic_top = add_pr_metrics(results_mimic_top)
results_mrmr_top = add_pr_metrics(results_mrmr_top)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


PANELS = [("auroc",  "ROC AUC",      0.5),      # (column, label, reference line)
          ("auprc",   "PR AUC",       None),     # baseline is per-cohort → no line
          ("pr_skill", "PR skill",     0.0)]      # 0 = chance
RUNS  = {"mimic100": results_mimic_top, "mrmr100": results_mrmr_top}
COLOR = {"mimic100": "#2a78d6", "mrmr100": "#eb6834"}

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.8), constrained_layout=True)

for ax, (col, label, ref) in zip(axes, PANELS):
    data = [RUNS[r][col].dropna().to_numpy() for r in RUNS]

    bp = ax.boxplot(data, positions=range(len(RUNS)), widths=0.5,
                    patch_artist=True, showfliers=False, zorder=2)
    for box, run in zip(bp["boxes"], RUNS):
        box.set(facecolor=COLOR[run], alpha=0.22, edgecolor=COLOR[run], linewidth=1.6)
    for part in ("whiskers", "caps"):
        for a, run in zip(bp[part], np.repeat(list(RUNS), 2)):
            a.set(color=COLOR[run], linewidth=1.2)
    for med, run in zip(bp["medians"], RUNS):
        med.set(color=COLOR[run], linewidth=2.2)

    for pos, (v, run) in enumerate(zip(data, RUNS)):
        ax.scatter(rng.normal(pos, 0.045, len(v)), v, s=10, color=COLOR[run],
                   alpha=0.35, zorder=3, linewidths=0)
        ax.text(pos, 0.02, f"med {np.median(v):.3f}", transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=8, color="#52514e")

    if ref is not None:
        ax.axhline(ref, color="#52514e", lw=1, ls=(0, (4, 3)), zorder=1)

    ax.set_xticks(range(len(RUNS)), list(RUNS))
    ax.set_title(label, loc="left", fontsize=12, pad=8)
    ax.grid(axis="y", lw=0.8, alpha=0.5, color="#e3e2dd"); ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("one point = one cohort")
fig.suptitle(f"mimic100 vs mrmr100 across {results_mimic_top.cohort_name.nunique()} cohorts "
             "(fold 0)", x=0.01, ha="left", fontsize=13)
plt.show()



In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr

RUNS = {
    "mimic100": results_mimic_top,
    "mrmr100": results_mrmr_top,
}
SELECTED_RUNS = ["mimic100"]
MODEL = "random_forest"

PREDICTORS = [("target_rate", "Positive rate"), ("n", "Cohort size")]
METRICS = [("auroc", "AUROC"), ("auprc", "AUPRC"), ("pr_skill", "PR skill")]

def prepare(df, model=MODEL):
    d = df[df.model == model] if "model" in df.columns else df
    if "n" not in d.columns and "n_cohort" in d.columns:
        d = d.assign(n=d["n_cohort"])
    cols = [c for c, _ in PREDICTORS] + [c for c, _ in METRICS]
    d = d[["cohort_name"] + cols].apply(pd.to_numeric, errors="coerce").assign(
        cohort_name=d["cohort_name"].values)
    return d.groupby("cohort_name", as_index=False)[cols].mean()

def _resid_ranks(a, b):
    ra, rb = pd.Series(a).rank().to_numpy(), pd.Series(b).rank().to_numpy()
    X = np.column_stack([np.ones_like(rb), rb])
    return ra - X @ np.linalg.lstsq(X, ra, rcond=None)[0]

def correlate_performance(df, alpha=0.05):
    d = prepare(df)
    rows = []
    for pcol, plabel in PREDICTORS:
        others = [c for c, _ in PREDICTORS if c != pcol]
        for mcol, mlabel in METRICS:
            sub = d[[pcol, mcol] + others].dropna()
            if len(sub) < 5:
                continue
            rs, ps = spearmanr(sub[pcol], sub[mcol])
            rp, pp = pearsonr(sub[pcol], sub[mcol])
            row = dict(predictor=plabel, metric=mlabel, n_cohorts=len(sub),
                       spearman=rs, p_spearman=ps, pearson=rp, p_pearson=pp)
            if others:
                z = others[0]
                row["spearman_partial"], row["p_partial"] = pearsonr(
                    _resid_ranks(sub[pcol], sub[z]), _resid_ranks(sub[mcol], sub[z]))
            rows.append(row)
    out = pd.DataFrame(rows)
    out["significant"] = out["p_spearman"] < alpha
    out["strength"] = pd.cut(out["spearman"].abs(), [-0.01, 0.1, 0.3, 0.5, 1.0],
                             labels=["negligible", "weak", "moderate", "strong"])
    return out

corr_summary = pd.concat(
    [correlate_performance(RUNS[r]).assign(run=r) for r in SELECTED_RUNS],
    ignore_index=True)



In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

COLOR = {"mimic100": "#2a78d6", "mrmr100": "#eb6834"}
REF = {"auroc": 0.5, "auprc": None, "pr_skill": 0.0, "pr_diff": 0.0, "pr_lift": 1.0}

nrows, ncols = len(PREDICTORS), len(METRICS) + 1
fig = plt.figure(figsize=(4.5 * ncols, 4.0 * nrows), constrained_layout=True)
gs = fig.add_gridspec(nrows, ncols)

d = prepare(RUNS[SELECTED_RUNS[0]])
ax0 = fig.add_subplot(gs[:, 0])
rho_c, p_c = spearmanr(d["n"], d["target_rate"])
ax0.scatter(d["n"], d["target_rate"], s=20, color="#7b3fb5", alpha=0.6, linewidths=0)
k = np.polyfit(np.log10(d["n"]), d["target_rate"], 1)
xs = np.linspace(d["n"].min(), d["n"].max(), 100)
ax0.plot(xs, np.polyval(k, np.log10(xs)), color="#7b3fb5", lw=1.8, ls="--")
ax0.set_xscale("log")
ax0.set_xlabel("Cohort size"); ax0.set_ylabel("Positive rate")
ax0.set_title(f"The confound: rho={rho_c:+.2f}, p={p_c:.1e}", loc="left", fontsize=11)
ax0.grid(lw=0.7, alpha=0.4, color="#e3e2dd"); ax0.set_axisbelow(True)
ax0.spines[["top", "right"]].set_visible(False)

for i, (pcol, plabel) in enumerate(PREDICTORS):
    for j, (mcol, mlabel) in enumerate(METRICS):
        ax = fig.add_subplot(gs[i, j + 1])
        for run in SELECTED_RUNS:
            dd = prepare(RUNS[run])[[pcol, mcol]].dropna()
            if len(dd) < 3:
                continue
            x, y = dd[pcol].to_numpy(), dd[mcol].to_numpy()
            c = COLOR.get(run, "#52514e")
            ax.scatter(x, y, s=18, color=c, alpha=0.55, linewidths=0)
            rho, p = spearmanr(x, y)
            logx = pcol == "n"
            k = np.polyfit(np.log10(x) if logx else x, y, 1)
            xs = np.linspace(x.min(), x.max(), 100)
            ax.plot(xs, np.polyval(k, np.log10(xs) if logx else xs), color=c, lw=1.6, ls="--")
            ax.plot([], [], " ", label=f"{run}: rho={rho:+.2f}, p={p:.1e}")
        ref = REF.get(mcol)
        if ref is not None:
            ax.axhline(ref, color="#52514e", lw=1, ls=(0, (4, 3)))
        if pcol == "n":
            ax.set_xscale("log")
        ax.set_xlabel(plabel); ax.set_ylabel(mlabel)
        ax.grid(lw=0.7, alpha=0.4, color="#e3e2dd"); ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False, fontsize=8, loc="best")

fig.suptitle(f"Performance vs cohort characteristics ({MODEL}, "
             f"{', '.join(SELECTED_RUNS)})", x=0.01, ha="left", fontsize=13)
plt.show()


In [ ]:
import pickle
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import rankdata, spearmanr
from tqdm import tqdm
from config.constants import PROJECT_ROOT

SAVED = Path(PROJECT_ROOT) / "saved_data"
DTYPES = {"subject_id": "int64", "hadm_id": "int64", "minute": "int64",
          "itemid": "string", "value": "float64"}

MODEL = "random_forest"
RUNS = {"mimic100": results_mimic_top, "mrmr100": results_mrmr_top}
HM_RUNS = ["mimic100", "mrmr100"]
HM_METRICS = [("auroc", "AUC-ROC"), ("pr_lift", "AUC-PR norm")]

FOLD, SEED, DAYS, FIRST_ADM = 0, 42, 14, True
K_POS, K_NEG, N_BOOT = 40, 40, 20

def load_fold_ids(cohort, fold=FOLD, seed=SEED, first_adm_only=FIRST_ADM):
    suffix = "_firstadm" if first_adm_only else ""
    f = SAVED / "folds" / cohort / f"seed_{seed}{suffix}" / f"fold_{fold}.pkl"
    if not f.exists():
        return None
    with open(f, "rb") as fh:
        tr, va, te = pickle.load(fh)
    return {"train": tr[:, 1], "val": va[:, 1], "test": te[:, 1]}

def load_split_features(cohort, fold=FOLD, days=DAYS, seed=SEED,
                        first_adm_only=FIRST_ADM, split="train",
                        feature_selection="mimic_top_100"):
    ids = load_fold_ids(cohort, fold, seed, first_adm_only)
    f = SAVED / f"features_{days}d" / cohort / "features.csv.gz"
    if ids is None or not f.exists():
        return None
    data = pd.read_csv(f, dtype=DTYPES)

    if feature_selection == "mimic_top_100":
        with open(Path(PROJECT_ROOT) / "data" / "top_features" / "mimic_top100_features.pkl", "rb") as fh:
            sel = list(map(str, pickle.load(fh)))
        data = data[data["itemid"].isin(sel)]
    elif feature_selection == "MRMR_top_100":
        method_dir = f"mrmr100_{days}d" + ("_firstadm" if first_adm_only else "")
        with open(SAVED / "features_selected_corr" / method_dir / cohort / f"fold_{fold}" / "selected_itemids.pkl", "rb") as fh:
            sel = list(map(str, pickle.load(fh)))
        data = data[data["itemid"].isin(sel)]

    train_present = np.intersect1d(ids["train"], data.hadm_id.unique())
    train_vars = data.loc[data.hadm_id.isin(train_present), "itemid"].unique()
    data = data[data["itemid"].isin(train_vars)]

    keep = np.intersect1d(ids[split], data.hadm_id.unique())
    return data[data.hadm_id.isin(keep)]

def wide_and_labels(cohort, **kw):
    d = load_split_features(cohort, **kw)
    if d is None or d.empty:
        return None, None
    wide = d.groupby(["hadm_id", "itemid"], observed=True)["value"].mean().unstack()
    lab = (pd.read_csv(SAVED / "cohorts" / "DTB" / "new" / f"{cohort}.csv.gz",
                       usecols=["hadm_id", "label"])
             .drop_duplicates("hadm_id").set_index("hadm_id")["label"])
    y = lab.reindex(wide.index)
    keep = y.notna()
    return wide[keep], y[keep].astype(int)

def merged_chars(run, model=MODEL):
    df = RUNS[run]
    d = df[df.model == model] if "model" in df.columns else df
    if "n" not in d.columns and "n_cohort" in d.columns:
        d = d.assign(n=d["n_cohort"])
    cols = ["target_rate", "n"] + [m for m, _ in HM_METRICS]
    d = d[["cohort_name"] + cols].groupby("cohort_name", as_index=False).mean()
    d = d.merge(feat_stats, on="cohort_name", how="inner")
    d["log_n"] = np.log10(d["n"])
    return d


In [ ]:
def cohort_missingness(cohort, **kw):
    wide, y = wide_and_labels(cohort, **kw)
    if wide is None:
        return None
    miss = wide.isna().mean()
    return {"cohort_name": cohort, "n_train": len(wide), "n_feats": wide.shape[1],
            "missingness_m": miss.mean(), "missingness_v": miss.var()}

FEAT_FILE = SAVED / "feature_stats_train_firstadm_fold0.csv"

if FEAT_FILE.exists():
    feat_stats = pd.read_csv(FEAT_FILE)
else:
    rows = [r for c in tqdm(rep_cohorts["cohort_name"])
            if (r := cohort_missingness(c)) is not None]
    feat_stats = pd.DataFrame(rows)
    feat_stats.to_csv(FEAT_FILE, index=False)

print(feat_stats.shape)
display(feat_stats.describe().round(3))


In [ ]:
def _auc_from_ranks(x, y):
    r = rankdata(x)
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def univariate_aucs(wide, y, min_obs=10):
    out = {}
    yy = y.to_numpy()
    for c in wide.columns:
        m = wide[c].notna().to_numpy()
        ys = yy[m]
        if m.sum() < min_obs or (ys == 1).sum() < 3 or (ys == 0).sum() < 3:
            continue
        out[c] = _auc_from_ranks(wide[c].to_numpy()[m], ys)
    return pd.Series(out, dtype=float)

def missing_aucs(wide, y):
    yy = y.to_numpy()
    if (yy == 1).sum() < 3 or (yy == 0).sum() < 3:
        return pd.Series(dtype=float)
    return pd.Series({c: _auc_from_ranks(wide[c].notna().astype(int).to_numpy(), yy)
                      for c in wide.columns}, dtype=float)

def cohort_signal_matched(cohort, k_pos=K_POS, k_neg=K_NEG, n_boot=N_BOOT, seed=0, **kw):
    wide, y = wide_and_labels(cohort, **kw)
    if wide is None:
        return None
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) < k_pos or len(neg_idx) < k_neg:
        return None

    rng = np.random.default_rng(seed)
    acc = []
    for _ in range(n_boot):
        sel = np.concatenate([rng.choice(pos_idx, k_pos, replace=False),
                              rng.choice(neg_idx, k_neg, replace=False)])
        w, yy = wide.iloc[sel], y.iloc[sel]
        s = (univariate_aucs(w, yy) - 0.5).abs() + 0.5
        sm = (missing_aucs(w, yy) - 0.5).abs() + 0.5
        if s.empty:
            continue
        acc.append({"auc_top5_m": s.nlargest(5).mean(), "auc_mean_m": s.mean(),
                    "n_auc_gt_65_m": (s >= 0.65).sum(),
                    "missauc_top5_m": sm.nlargest(5).mean() if len(sm) else np.nan})
    if not acc:
        return None
    out = pd.DataFrame(acc).mean().to_dict()
    out.update({"cohort_name": cohort,
                "n_pos_train": len(pos_idx), "n_neg_train": len(neg_idx)})
    return out

SIGNAL_FILE = SAVED / f"signal_stats_matched_k{K_POS}_train_firstadm_fold0.csv"

if SIGNAL_FILE.exists():
    signal_stats = pd.read_csv(SIGNAL_FILE)
else:
    rows, dropped = [], []
    for c in tqdm(rep_cohorts["cohort_name"]):
        r = cohort_signal_matched(c)
        rows.append(r) if r is not None else dropped.append(c)
    signal_stats = pd.DataFrame(rows)
    signal_stats.to_csv(SIGNAL_FILE, index=False)
    print(f"dropped {len(dropped)} cohorts with < {K_POS} train pos or < {K_NEG} train neg")

feat_stats = feat_stats.drop(columns=[c for c in signal_stats.columns
                                      if c != "cohort_name" and c in feat_stats.columns])
feat_stats = feat_stats.merge(signal_stats, on="cohort_name", how="left")

d_chk = merged_chars("mimic100")
print(f"\nbias check ({len(d_chk)} cohorts):")
for c in ["auc_top5_m", "auc_mean_m", "n_auc_gt_65_m", "missauc_top5_m",
          "missingness_m", "missingness_v"]:
    v = d_chk[[c]].assign(n_pos=d_chk.n * d_chk.target_rate).dropna()
    rho, p = spearmanr(v[c], v.n_pos)
    print(f"  {c:16s} vs n_pos: rho={rho:+.3f}  p={p:.1e}")

display(signal_stats.describe().round(3))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory

CHARS = [("target_rate", "Positive rate"), ("log_n", "Log cohort size"),
         ("auc_top5_m", "Top-5 lab AUC"), ("auc_mean_m", "Mean lab AUC"),
         ("n_auc_gt_65_m", "# labs AUC>=0.65"),
         ("missauc_top5_m", "Top-5 missingness AUC"),
         ("missingness_m", "Missingness M"), ("missingness_v", "Missingness V")]

def sig_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

cols = [f"{r}_{m}" for r in HM_RUNS for m, _ in HM_METRICS]
corr = pd.DataFrame(index=[c for c, _ in CHARS], columns=cols, dtype=float)
stars = pd.DataFrame(index=[c for c, _ in CHARS], columns=cols, dtype=str)

for r in HM_RUNS:
    d = merged_chars(r)
    for m, _ in HM_METRICS:
        for c, _ in CHARS:
            v = d[[c, m]].dropna()
            rho, p = spearmanr(v[c], v[m])
            corr.loc[c, f"{r}_{m}"] = rho
            stars.loc[c, f"{r}_{m}"] = sig_stars(p)

fig, ax = plt.subplots(figsize=(2.2 * len(cols) + 3, 0.72 * len(CHARS) + 2.4))
fig.subplots_adjust(top=0.82)
vals = corr.values.astype(float)
im = ax.imshow(vals, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

for i in range(len(CHARS)):
    for j in range(len(cols)):
        ax.text(j, i, f"{vals[i, j]:.2f}{stars.iloc[i, j]}", ha="center", va="center",
                fontsize=11, color="black" if abs(vals[i, j]) < 0.6 else "white")

ax.set_xticks(range(len(cols)))
ax.set_xticklabels([lab for _ in HM_RUNS for _, lab in HM_METRICS], fontsize=11)
ax.set_yticks(range(len(CHARS)))
ax.set_yticklabels([lab for _, lab in CHARS], fontsize=13)
ax.tick_params(axis="x", bottom=False); ax.tick_params(axis="y", left=False)

trans = blended_transform_factory(ax.transData, ax.transAxes)
for idx, r in enumerate(HM_RUNS):
    center = idx * len(HM_METRICS) + (len(HM_METRICS) - 1) / 2
    ax.text(center, 1.02, r, ha="center", va="bottom", fontsize=13, transform=trans)
for idx in range(1, len(HM_RUNS)):
    ax.axvline(x=idx * len(HM_METRICS) - 0.5, color="white", linewidth=2)

for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"Spearman: cohort characteristics vs performance ({MODEL})",
             fontsize=15, pad=32)

cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_ticks([-1, -0.5, 0, 0.5, 1]); cbar.outline.set_visible(False)
plt.show()


In [ ]:
#Positive rate (−0.08 ns / −0.45***) — no effect on AUROC; the strong AUC-PR number is the metric's own prevalence baseline, not a finding.
#Log cohort size (+0.17 ns / +0.39***) — no effect on AUROC; the AUC-PR number is prevalence in disguise (size and prevalence correlate −0.68).
#Top-5 lab AUC (+0.12 ns) — the single best labs don't matter; a max over 100 noisy estimates is mostly luck.
#Mean lab AUC (+0.46*** / +0.35***) — the main finding: cohorts whose labs are individually informative produce better models.
# labs AUC ≥ 0.65 (+0.41*** / +0.33**) — same finding from the other side: what matters is how many labs carry signal, i.e. breadth, not peak strength.
#Top-5 missingness AUC (+0.37*** / +0.33**) — whether a lab was ordered predicts the outcome; informative missingness is real signal, and justifies the mask channel in VMD.
#Missingness M (−0.08 ns) — the amount of missing data is irrelevant.
#Missingness V (+0.01 ns) — how unevenly missingness spreads across labs is irrelevant.

# Select cohorts based on CI of ROC

In [ ]:
# TODO: will come back to this later if we want to technically select cohorts

In [ ]:
chars = pd.read_csv(cohorts_dir / "first_adms_cohorts_summary.csv") 

ASSUMED_AUC = 0.60          # conservative; SE is largest at low AUROC
TARGET_CI_HALFWIDTH = 0.2  # decide this from what you need to detect

npos, nneg = chars['n_pos_test_estimate'], chars['n_neg_test_estimate']
Q1 = ASSUMED_AUC/(2-ASSUMED_AUC); Q2 = 2*ASSUMED_AUC**2/(1+ASSUMED_AUC)
var = (ASSUMED_AUC*(1-ASSUMED_AUC) + (npos-1)*(Q1-ASSUMED_AUC**2)
       + (nneg-1)*(Q2-ASSUMED_AUC**2))/(npos*nneg)
chars['expected_se'] = np.sqrt(var)
chars['expected_ci_halfwidth'] = 1.96 * chars['expected_se']
chars['worth_training'] = chars['expected_ci_halfwidth'] <= TARGET_CI_HALFWIDTH

print(chars['worth_training'].value_counts())


In [ ]:
print(pd.crosstab(chars["reliable"], chars["worth_training"], margins=True))
